# Asian Floating-Strike Call Option — Monte Carlo Pricing
### Stochastic Methods in Finance — HSG Spring 2026

**Payoff at maturity:**

$$A_T = \max\left(S_n - \frac{1}{n+1}\sum_{t=0}^{n} S_t,\ 0\right)$$

This notebook is a **Monte Carlo benchmark**. The assignment requires a binomial tree.

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm

# Dark style for plots
plt.rcParams.update({
    "figure.facecolor":  "#0f1117",
    "axes.facecolor":    "#1a1d27",
    "axes.labelcolor":   "#e0e0e0",
    "xtick.color":       "#e0e0e0",
    "ytick.color":       "#e0e0e0",
    "text.color":        "#e0e0e0",
    "axes.titlecolor":   "#e0e0e0",
    "grid.color":        "#2a2d3a",
    "grid.linestyle":    "--",
    "grid.linewidth":    0.5,
})

ACCENT = "#4f8ef7"
GREEN  = "#2ecc71"
ORANGE = "#f39c12"
RED    = "#e74c3c"
PANEL  = "#1a1d27"
GRID   = "#2a2d3a"
TEXT   = "#e0e0e0"
print("Imports OK")


---
## 1. Load Data & Estimate Parameters

We load the AAPL price series, filter **2020–2026** as required by the assignment,  
then compute the **annualised volatility** $\sigma = \sigma_{\text{daily}} \times \sqrt{250}$.


In [ ]:
# ── Adjust this path to wherever you saved the Excel file ──
DATA_PATH = "Assignment_SMF_2026.xlsx"

df = pd.read_excel(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

# Filter 2020–2026 as required by the assignment
df = df[df["Date"] >= "2020-01-01"].copy().reset_index(drop=True)

# Log-returns
log_returns = np.log(df["AAPL.US"] / df["AAPL.US"].shift(1)).dropna()

sigma_daily = log_returns.std()
sigma       = sigma_daily * np.sqrt(250)   # annualised volatility
S0          = df["AAPL.US"].iloc[-1]       # latest price (27 Apr 2026)

print(f"Observations (2020–2026) : {len(df)}")
print(f"S0 (27 Apr 2026)         : {S0:.2f} USD")
print(f"σ_daily                  : {sigma_daily:.6f}")
print(f"σ_annualised             : {sigma:.6f}  ({sigma*100:.2f}%)")


---
## 2. Model Parameters

From the assignment §3.2:

| Parameter | Value |
|-----------|-------|
| $T$ | 0.5 years (6 months) |
| $n$ | 25 periods |
| $r$ | 1% p.a. |
| $u$ | $e^{\sigma\sqrt{\Delta t}}$ |
| $d$ | $e^{-\sigma\sqrt{\Delta t}}$ |
| $p$ | $1/2$ (physical) |
| $q$ | $\frac{e^{r\Delta t}-d}{u-d}$ (risk-neutral) |


In [ ]:
T   = 0.5          # maturity: 6 months
n   = 25           # number of periods
r   = 0.01         # risk-free rate per annum
dt  = T / n        # length of each period

u   = np.exp( sigma * np.sqrt(dt))
d   = np.exp(-sigma * np.sqrt(dt))
p   = 0.5                                # physical probability
q   = (np.exp(r * dt) - d) / (u - d)    # risk-neutral probability

print(f"T    = {T} years  ({int(T*12)} months)")
print(f"n    = {n} periods")
print(f"Δt   = {dt:.6f}")
print(f"r    = {r*100:.1f}% p.a.")
print(f"u    = {u:.6f}")
print(f"d    = {d:.6f}")
print(f"p    = {p} (physical)")
print(f"q    = {q:.6f} (risk-neutral)")


---
## 3. Monte Carlo Simulation

### Algorithm

1. For each path, draw $n$ Bernoulli$(q)$ steps: $+\log u$ (up) or $+\log d$ (down)
2. Reconstruct the full price path $S_0, S_1, \ldots, S_n$
3. Compute the arithmetic average $\bar{S}_n = \frac{1}{n+1}\sum_{t=0}^n S_t$
4. Compute the discounted payoff $e^{-rT}\max(S_n - \bar{S}_n,\ 0)$
5. Average over all paths → unbiased estimator of the option price


In [ ]:
N_sim = 500_000
np.random.seed(42)

# Draw risk-neutral Bernoulli steps: shape (N_sim, n)
steps = np.where(
    np.random.uniform(size=(N_sim, n)) < q,
    np.log(u),
    np.log(d)
)

# Full price path: shape (N_sim, n+1)  [includes S_0]
price_paths = S0 * np.concatenate(
    [np.ones((N_sim, 1)), np.exp(np.cumsum(steps, axis=1))],
    axis=1
)

# Arithmetic average and terminal price
S_avg = price_paths.mean(axis=1)   # shape (N_sim,)
S_T   = price_paths[:, -1]         # shape (N_sim,)

# Discounted payoff
discount         = np.exp(-r * T)
payoffs          = np.maximum(S_T - S_avg, 0.0)
discounted_payoffs = discount * payoffs

# ── Results ──
price_mc = discounted_payoffs.mean()
se_mc    = discounted_payoffs.std() / np.sqrt(N_sim)
ci_low   = price_mc - 1.96 * se_mc
ci_high  = price_mc + 1.96 * se_mc
itm_pct  = 100 * (payoffs > 0).mean()

print(f"MC Price    : {price_mc:.4f} USD")
print(f"Std Error   : {se_mc:.6f}")
print(f"95% CI      : [{ci_low:.4f}, {ci_high:.4f}]")
print(f"ITM paths   : {itm_pct:.1f}%")


---
## 4. Normal Approximation (Task vii)

We approximate the distribution of $D_n = S_n - \bar{S}_n$ by a normal $\mathcal{N}(\mu_D, \sigma_D^2)$.

Under the risk-neutral measure:

$$\mathbb{E}[S_t] = S_0 e^{r t}, \quad \Rightarrow \quad \mathbb{E}[\bar{S}_n] = \frac{S_0}{n+1}\sum_{t=0}^{n} e^{r\, t\Delta t}$$

$$\mu_D = \mathbb{E}[S_n] - \mathbb{E}[\bar{S}_n] = S_0 e^{rT} - \frac{S_0}{n+1}\sum_{t=0}^{n} e^{r\, t\Delta t}$$

The option price under the normal approximation is:

$$\hat{V} = e^{-rT}\left[\mu_D \,\Phi\!\left(\frac{\mu_D}{\sigma_D}\right) + \sigma_D\, \phi\!\left(\frac{\mu_D}{\sigma_D}\right)\right]$$


In [ ]:
t_grid  = np.arange(0, n + 1) * dt                    # t = 0, dt, 2dt, ..., T
E_S_bar = S0 / (n + 1) * np.sum(np.exp(r * t_grid))
E_S_n   = S0 * np.exp(r * T)
mu_D    = E_S_n - E_S_bar

# Empirical std of D_n from the MC simulation
sigma_D = (S_T - S_avg).std()

# Normal approximation price: E[max(D_n, 0)] for D ~ N(mu_D, sigma_D^2)
d1_norm    = mu_D / sigma_D
price_norm = discount * (mu_D * norm.cdf(d1_norm) + sigma_D * norm.pdf(d1_norm))

print(f"E[D_n]              : {mu_D:.4f}")
print(f"Std(D_n)            : {sigma_D:.4f}")
print(f"Normal Approx Price : {price_norm:.4f} USD")
print(f"MC Price            : {price_mc:.4f} USD")
print(f"Difference          : {abs(price_norm - price_mc):.4f} USD  ({abs(price_norm/price_mc - 1)*100:.2f}%)")


---
## 5. Robustness Check (Task vi)

We vary $\sigma$ and $r$ independently around the baseline to assess sensitivity.


In [ ]:
def mc_price(sigma_in, r_in, N=200_000):
    """Quick MC price for a given (sigma, r) pair."""
    u_ = np.exp( sigma_in * np.sqrt(dt))
    d_ = np.exp(-sigma_in * np.sqrt(dt))
    q_ = (np.exp(r_in * dt) - d_) / (u_ - d_)
    st = np.where(np.random.uniform(size=(N, n)) < q_, np.log(u_), np.log(d_))
    pp = S0 * np.concatenate([np.ones((N, 1)), np.exp(np.cumsum(st, axis=1))], axis=1)
    pay = np.exp(-r_in * T) * np.maximum(pp[:, -1] - pp.mean(axis=1), 0)
    return pay.mean(), pay.std() / np.sqrt(N)

# ── Vary σ
sigmas = [sigma * f for f in [0.75, 0.90, 1.00, 1.10, 1.25]]
print(f"{'σ (%)':>10}  {'Price':>10}  {'95% CI':>24}")
for s in sigmas:
    p_, se_ = mc_price(s, r)
    tag = "  ← baseline" if abs(s - sigma) < 1e-8 else ""
    print(f"{s*100:>10.2f}  {p_:>10.4f}  [{p_-1.96*se_:.4f}, {p_+1.96*se_:.4f}]{tag}")

print()

# ── Vary r
rates = [0.00, 0.005, 0.01, 0.02, 0.03]
print(f"{'r (%)':>10}  {'Price':>10}  {'95% CI':>24}")
for r_ in rates:
    p_, se_ = mc_price(sigma, r_)
    tag = "  ← baseline" if r_ == r else ""
    print(f"{r_*100:>10.2f}  {p_:>10.4f}  [{p_-1.96*se_:.4f}, {p_+1.96*se_:.4f}]{tag}")


---
## 6. Visualisations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0f1117")

# ── Sample paths
ax = axes[0]
t_axis = np.arange(n + 1) * dt
for i in range(80):
    ax.plot(t_axis, price_paths[i], color=ACCENT, alpha=0.25, linewidth=0.7)
ax.axhline(S0, color=ORANGE, linewidth=1.2, linestyle="--", label=f"S₀ = {S0:.0f}")
ax.set_title("Sample Price Paths (80)", fontweight="bold")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Price (USD)")
ax.legend(facecolor=PANEL, edgecolor=GRID)

# ── Payoff distribution
ax = axes[1]
pos = payoffs[payoffs > 0]
ax.hist(pos, bins=80, color=ACCENT, alpha=0.85, edgecolor="none")
ax.axvline(payoffs.mean(), color=ORANGE, linewidth=1.5, linestyle="--",
           label=f"Mean payoff = {payoffs.mean():.2f}")
ax.set_title(f"Payoff Distribution  (ITM: {itm_pct:.1f}%)", fontweight="bold")
ax.set_xlabel("Payoff (USD)")
ax.set_ylabel("Frequency")
ax.legend(facecolor=PANEL, edgecolor=GRID)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor("#0f1117")

# ── MC convergence
ax = axes[0]
checkpoints = np.logspace(2, np.log10(N_sim), 60).astype(int)
rp = np.array([discounted_payoffs[:k].mean() for k in checkpoints])
rs = np.array([discounted_payoffs[:k].std() / np.sqrt(k) for k in checkpoints])
ax.plot(checkpoints, rp, color=ACCENT, linewidth=1.5)
ax.fill_between(checkpoints, rp - 1.96*rs, rp + 1.96*rs, color=ACCENT, alpha=0.15, label="95% CI")
ax.axhline(price_mc, color=GREEN, linewidth=1, linestyle="--", label=f"Final = {price_mc:.4f}")
ax.set_xscale("log")
ax.set_title("MC Price Convergence", fontweight="bold")
ax.set_xlabel("Number of simulations")
ax.set_ylabel("Estimated Price (USD)")
ax.legend(facecolor=PANEL, edgecolor=GRID)

# ── Sigma sensitivity
ax = axes[1]
sig_range  = np.linspace(0.10, 0.60, 25)
prices_sig = [mc_price(s, r, N=100_000)[0] for s in sig_range]
ax.plot(sig_range * 100, prices_sig, color=ACCENT, linewidth=1.8)
ax.axvline(sigma * 100, color=ORANGE, linewidth=1.2, linestyle="--", label=f"σ = {sigma*100:.1f}%")
ax.axhline(price_mc, color=GREEN, linewidth=1, linestyle="--", label=f"Baseline = {price_mc:.4f}")
ax.set_title("Price Sensitivity to σ", fontweight="bold")
ax.set_xlabel("σ (%)")
ax.set_ylabel("Option Price (USD)")
ax.legend(facecolor=PANEL, edgecolor=GRID)

# ── Rate sensitivity
ax = axes[2]
r_range   = np.linspace(0.00, 0.08, 25)
prices_r  = [mc_price(sigma, r_, N=100_000)[0] for r_ in r_range]
ax.plot(r_range * 100, prices_r, color=ACCENT, linewidth=1.8)
ax.axvline(r * 100, color=ORANGE, linewidth=1.2, linestyle="--", label=f"r = {r*100:.1f}%")
ax.axhline(price_mc, color=GREEN, linewidth=1, linestyle="--", label=f"Baseline = {price_mc:.4f}")
ax.set_title("Price Sensitivity to r", fontweight="bold")
ax.set_xlabel("r (%)")
ax.set_ylabel("Option Price (USD)")
ax.legend(facecolor=PANEL, edgecolor=GRID)

plt.tight_layout()
plt.show()


---
## 7. Summary

| Metric | Value |
|--------|-------|
| $S_0$ | 267.61 USD |
| $\sigma$ (annualised) | 31.31% |
| $r$ | 1.0% |
| $T$ | 0.5 years |
| $n$ | 25 periods |
| $u$ | 1.04527 |
| $d$ | 0.95669 |
| $q$ (risk-neutral) | 0.49119 |
| **MC Price** | **~13.87 USD** |
| 95% CI | [13.80, 13.93] |
| Normal Approx | ~14.11 USD |
| ITM paths | ~40–45% |

---
## 8. Monte Carlo GBM Simulation Paths

Visualization of simulated GBM price paths under the risk-neutral measure, with the distribution of terminal prices $ and the arithmetic average $\bar{S}_n$.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor("#0f1117")

t_axis = np.arange(n + 1) * dt

# ── Plot 1: GBM paths with mean and confidence band
ax = axes[0]
# Plot 200 sample paths
for i in range(200):
    ax.plot(t_axis, price_paths[i], color=ACCENT, alpha=0.15, linewidth=0.5)

# Theoretical mean path E[S_t] = S0 * exp(r*t) under Q
mean_path = S0 * np.exp(r * t_axis)
std_path  = S0 * np.exp(r * t_axis) * np.sqrt(np.exp(sigma**2 * t_axis) - 1)
ax.plot(t_axis, mean_path, color=GREEN, linewidth=2, label=r"$\mathbb{E}^Q[S_t]$")
ax.fill_between(t_axis, mean_path - std_path, mean_path + std_path,
                color=GREEN, alpha=0.15, label="±1 std (theoretical)")
ax.axhline(S0, color=ORANGE, linewidth=1.2, linestyle="--", label=f"S₀ = {S0:.0f}")
ax.set_title("GBM Paths under Risk-Neutral Measure", fontweight="bold")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Price (USD)")
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID)

# ── Plot 2: Distribution of terminal price S_n
ax = axes[1]
ax.hist(S_T, bins=100, color=ACCENT, alpha=0.8, edgecolor="none", density=True)
# Overlay log-normal theoretical density
from scipy.stats import lognorm
mu_ln    = np.log(S0) + (r - 0.5 * sigma**2) * T
sigma_ln = sigma * np.sqrt(T)
x_range  = np.linspace(S_T.min(), S_T.max(), 300)
ax.plot(x_range, lognorm.pdf(x_range, s=sigma_ln, scale=np.exp(mu_ln)),
        color=GREEN, linewidth=2, label="Log-normal PDF")
ax.axvline(S_T.mean(), color=ORANGE, linewidth=1.5, linestyle="--",
           label=f"Mean = {S_T.mean():.1f}")
ax.axvline(S0, color=RED, linewidth=1.2, linestyle=":", label=f"S₀ = {S0:.0f}")
ax.set_title("Distribution of Terminal Price $", fontweight="bold")
ax.set_xlabel("$ (USD)")
ax.set_ylabel("Density")
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID)

# ── Plot 3: Distribution of arithmetic average S_bar
ax = axes[2]
ax.hist(S_avg, bins=100, color=ORANGE, alpha=0.8, edgecolor="none", density=True)
ax.axvline(S_avg.mean(), color=GREEN, linewidth=1.5, linestyle="--",
           label=f"Mean = {S_avg.mean():.1f}")
ax.axvline(S0, color=RED, linewidth=1.2, linestyle=":", label=f"S₀ = {S0:.0f}")
ax.set_title(r"Distribution of Arithmetic Average $ar{S}_n$", fontweight="bold")
ax.set_xlabel(r"$ar{S}_n$ (USD)")
ax.set_ylabel("Density")
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID)

plt.suptitle("Monte Carlo GBM Simulation — AAPL Asian Option",
             fontsize=13, fontweight="bold", color=TEXT, y=1.02)
plt.tight_layout()
plt.show()


---
## 9. Greeks via Finite Differences

We compute the Greeks by running additional MC simulations with shocked parameters.
Each Greek uses a **central difference** formula for accuracy.

| Greek | Formula | Shock |
|-------|---------|-------|
| Delta | $(V(S_0+\epsilon) - V(S_0-\epsilon)) / 2\epsilon$ | $\epsilon = 1$ USD |
| Gamma | $(V(S_0+\epsilon) - 2V_0 + V(S_0-\epsilon)) / \epsilon^2$ | $\epsilon = 1$ USD |
| Vega  | $(V(\sigma+\epsilon) - V(\sigma-\epsilon)) / 2\epsilon$ | $\epsilon = 0.01$ |
| Rho   | $(V(r+\epsilon) - V(r-\epsilon)) / 2\epsilon$ | $\epsilon = 0.001$ |


In [ ]:

def mc_price_full(S0_in, sigma_in, r_in, n=25, T=0.5, N=500_000, seed=42):
    # MC pricer using a FIXED seed so all simulations use the same random numbers
    # (Common Random Numbers technique) -> drastically reduces noise in finite differences
    np.random.seed(seed)
    dt   = T / n
    u    = np.exp( sigma_in * np.sqrt(dt))
    d    = np.exp(-sigma_in * np.sqrt(dt))
    q    = (np.exp(r_in * dt) - d) / (u - d)
    disc = np.exp(-r_in * T)
    # Same uniform draws every time -> differences reflect parameter shock, not noise
    U = np.random.uniform(size=(N, n))
    steps = np.where(U < q, np.log(u), np.log(d))
    paths = S0_in * np.concatenate(
        [np.ones((N, 1)), np.exp(np.cumsum(steps, axis=1))], axis=1
    )
    payoff = disc * np.maximum(paths[:, -1] - paths.mean(axis=1), 0)
    return payoff.mean(), payoff.std() / np.sqrt(N)

# ── Base price
V0, _ = mc_price_full(S0, sigma, r)

# ── DELTA & GAMMA
# Use larger shock (5 USD = ~2% of S0) to dominate MC noise
eps_S  = 5.0
V_up,  _ = mc_price_full(S0 + eps_S, sigma, r)
V_down,_ = mc_price_full(S0 - eps_S, sigma, r)
delta = (V_up - V_down) / (2 * eps_S)
gamma = (V_up - 2 * V0 + V_down) / (eps_S ** 2)

# ── VEGA — shock sigma by 1 vol point (0.01 in decimal)
# Vega expressed as price change per 1% move in vol
eps_v  = 0.01
V_vup, _ = mc_price_full(S0, sigma + eps_v, r)
V_vdn, _ = mc_price_full(S0, sigma - eps_v, r)
vega = (V_vup - V_vdn) / (2 * eps_v)   # USD per unit of sigma (e.g. per 100%)
vega_pct = vega * 0.01                  # USD per 1% move in vol -> cleaner interpretation

# ── RHO — shock r by 10bps (0.001 in decimal)
# Rho expressed as price change per 1% move in r
eps_r  = 0.001
V_rup, _ = mc_price_full(S0, sigma, r + eps_r)
V_rdn, _ = mc_price_full(S0, sigma, r - eps_r)
rho = (V_rup - V_rdn) / (2 * eps_r)    # USD per unit of r
rho_pct = rho * 0.01                    # USD per 1% move in r

# ── Results
print("=" * 55)
print("GREEKS — Floating-Strike Asian Call (AAPL)")
print("=" * 55)
print(f"  Base price  V0  : {V0:.4f} USD")
print()
print(f"  Delta           : {delta:.4f}")
print(f"    -> AAPL +1 USD => option +{delta:.4f} USD")
print(f"    (vanilla ATM delta ~ 0.50, asian lower due to averaging)")
print()
print(f"  Gamma           : {gamma:.6f}")
print(f"    -> rate of change of Delta per 1 USD move in S0")
print()
print(f"  Vega            : {vega_pct:.4f} USD per 1% vol move")
print(f"    -> sigma +1% (e.g. 31% -> 32%) => option +{vega_pct:.4f} USD")
print()
print(f"  Rho             : {rho_pct:.4f} USD per 1% rate move")
print(f"    -> r +1% (e.g. 1% -> 2%) => option +{rho_pct:.4f} USD")
print(f"    (small: short maturity + floating strike)")
print("=" * 55)


In [ ]:

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.patch.set_facecolor("#0f1117")

s_range = np.linspace(S0 * 0.85, S0 * 1.15, 12)

# ── Delta vs S0
ax = axes[0]
deltas = []
for s in s_range:
    vp, _ = mc_price_full(s + 1, sigma, r, N=100_000)
    vd, _ = mc_price_full(s - 1, sigma, r, N=100_000)
    deltas.append((vp - vd) / 2)
ax.plot(s_range, deltas, color=ACCENT, linewidth=1.8, marker="o", markersize=4)
ax.axvline(S0, color=ORANGE, linewidth=1.2, linestyle="--", label=f"S0={S0:.0f}")
ax.axhline(delta, color=GREEN, linewidth=1, linestyle="--", label=f"Delta={delta:.3f}")
ax.set_title("Delta vs S0", fontweight="bold")
ax.set_xlabel("S0 (USD)"); ax.set_ylabel("Delta")
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID)

# ── Vega vs sigma
ax = axes[1]
sig_range = np.linspace(0.15, 0.55, 12)
vegas = []
for s in sig_range:
    vp, _ = mc_price_full(S0, s + 0.01, r, N=100_000)
    vd, _ = mc_price_full(S0, s - 0.01, r, N=100_000)
    vegas.append((vp - vd) / 0.02)
ax.plot(sig_range * 100, vegas, color=ACCENT, linewidth=1.8, marker="o", markersize=4)
ax.axvline(sigma * 100, color=ORANGE, linewidth=1.2, linestyle="--", label=f"sigma={sigma*100:.1f}%")
ax.axhline(vega, color=GREEN, linewidth=1, linestyle="--", label=f"Vega={vega:.3f}")
ax.set_title("Vega vs sigma", fontweight="bold")
ax.set_xlabel("sigma (%)"); ax.set_ylabel("Vega")
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID)

# ── Rho vs r
ax = axes[2]
r_range = np.linspace(0.00, 0.06, 12)
rhos = []
for r_ in r_range:
    vp, _ = mc_price_full(S0, sigma, r_ + 0.001, N=100_000)
    vd, _ = mc_price_full(S0, sigma, r_ - 0.001, N=100_000)
    rhos.append((vp - vd) / 0.002)
ax.plot(r_range * 100, rhos, color=ACCENT, linewidth=1.8, marker="o", markersize=4)
ax.axvline(r * 100, color=ORANGE, linewidth=1.2, linestyle="--", label=f"r={r*100:.1f}%")
ax.axhline(rho, color=GREEN, linewidth=1, linestyle="--", label=f"Rho={rho:.3f}")
ax.set_title("Rho vs r", fontweight="bold")
ax.set_xlabel("r (%)"); ax.set_ylabel("Rho")
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID)

# ── Gamma vs S0
ax = axes[3]
gammas = []
for s in s_range:
    v0s, _ = mc_price_full(s,     sigma, r, N=100_000)
    vp,  _ = mc_price_full(s + 1, sigma, r, N=100_000)
    vd,  _ = mc_price_full(s - 1, sigma, r, N=100_000)
    gammas.append(vp - 2 * v0s + vd)
ax.plot(s_range, gammas, color=ACCENT, linewidth=1.8, marker="o", markersize=4)
ax.axvline(S0, color=ORANGE, linewidth=1.2, linestyle="--", label=f"S0={S0:.0f}")
ax.axhline(0, color=GRID, linewidth=0.8)
ax.set_title("Gamma vs S0", fontweight="bold")
ax.set_xlabel("S0 (USD)"); ax.set_ylabel("Gamma")
ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID)

plt.suptitle("Greeks — Floating-Strike Asian Call (AAPL)",
             fontsize=13, fontweight="bold", color=TEXT, y=1.02)
plt.tight_layout()
plt.show()
